<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Databricks Performance Optimization
</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">03 Demo - Data Layout: Small Files and Self-Tuning Tables</span>
    </div>
</div>

![DB Academy](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/databricks_academy_logo.png)

# Demo - Data Layout: Small Files and Self-Tuning Tables
#### Duration: 20 minutes

## Overview

The small-file problem hasn't changed in years: thousands of tiny files (or a few oversized ones) throttle scans, shuffles, and DML. On Unity Catalog managed tables, the platform now fixes this automatically. 

On Unity Catalog managed tables, the layout tunes itself. Optimized writes right-size files as you write, auto compaction consolidates them, file sizes autotune to the table's size, and Predictive Optimization runs `OPTIMIZE` for you. You verify the result with `DESCRIBE DETAIL` instead of tuning the layout by hand.

This demo puts the two worlds side by side. You will stage a real file explosion the legacy way (plain writes to a non-managed path), read the damage with `DESCRIBE DETAIL`, then write the same data to a managed table and watch it right-size itself with no configuration. Along the way you will see why partitioning is now the exception, not the default, and which legacy tuning knobs to retire.

> This demo applies the data-layout techniques introduced in Part 1. You prove every claim with `DESCRIBE DETAIL` (file count and size), following the same measurement approach from Part 2.

## Learning Objectives

By the end of this demo, you will be able to:

1. **Explain the small-file problem**: where it comes from and why it still throttles scans, shuffles, and DML.
2. **Stage and measure** a file explosion, and read `numFiles` and `sizeInBytes` from `DESCRIBE DETAIL`.
3. **Describe how managed tables self-tune**: optimized writes, auto compaction, file-size autotuning, and ingestion-time clustering.
4. **Explain how Predictive Optimization runs `OPTIMIZE`** for managed tables and how to verify it ran.
5. **Apply the partitioning-is-the-exception rule** and retire the legacy file-size tuning knobs.

<div style="border-left:4px solid #FFAB00;background:rgba(255,171,0,0.08);padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#B8860B;margin-bottom:6px;font-size:15pt;">Pre-Run Timing Note</strong>
  <div style="color:#0b2026;font-size:14pt;line-height:1.55;">
    This notebook reads from <code>system_vw.storage.predictive_optimization_operations_history</code>, which lags live activity by <strong>up to about 2 hours</strong>. 
    <br></br>
    Predictive Optimization is also asynchronous, so it may not have run on your managed table yet. If you want live results, create the managed table well in advance so Predictive Optimization has time to act. The <code>po_operations()</code> helper guards against empty results with a descriptive message.
  </div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select a Compute Environment</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**

![Serverless Select](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook requires **Python and was tested using Serverless V5**. The entire demo (both the "before" legacy path and the "after" managed table path) runs on serverless. Other compute may work but is not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## A. Classroom Setup

Run the setup below. It builds the shared retail demo tables (idempotent) and stages an exploded, non-managed small-files copy of **orders** at a Volume path. It exposes `SMALL_FILES_PATH` (the "before") and `MANAGED_TABLE` (the "after", which you create live).

In [ ]:
%run ./Includes/Classroom-Setup-03


## B. The Small-File Problem
<br />
<div class="bnr-wrap">
<style>
.bnr-wrap { font-family: sans-serif; max-width: 1100px; margin: 0 auto; display: flex; flex-direction: column; gap: 20px; }
.bnr-wrap * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.danger { --bnr-bg: rgba(255,95,70,0.10); --bnr-accent: #FF5F46; --bnr-icon-fg: #FFFFFF; --bnr-title: #98102A; --bnr-tagline: #5E7077; }
</style>

<div class="ds-banner danger">
  <div class="ds-banner-icon">!</div>
  <div>
    <div class="ds-banner-title">Too many small files throttle scans, shuffles and operations</div>
    <div class="ds-banner-tagline" style="padding-top:16px">Thousands of tiny files become thousands of tiny tasks, so time goes to scheduling and I/O overhead instead of processing data.</div>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Where small files come from
- High-parallelism writes that emit one file per task (default Spark behavior without optimized writes).
- Frequent small appends or streaming microbatches, each commit writes at least one file per partition.
- Over-repartitioning, or partitioning a table on a high-cardinality column so each partition holds almost nothing.
- Many small `MERGE`, `UPDATE`, or `DELETE` operations over time on a non-managed table that is never compacted. (On managed tables, these operations always auto-compact.)

### Why it still throttles everything
- **Reads**: the engine packs file bytes into tasks up to 128 MB (`spark.sql.files.maxPartitionBytes`) and charges a per-file open cost (`spark.sql.files.openCostInBytes`, about 4 MB). Thousands of tiny files defeat the packing and produce thousands of tiny tasks, so time goes to scheduling and opening files, not to actual data processing. These configs are shown for understanding only: on serverless compute they are managed and fixed by the platform, so you do not tune them. They are adjustable only on classic or open-source Spark compute, which is outside the scope of this course.
- **Shuffles and DML**: more files means more metadata to track, more listing overhead, and more per-file coordination cost.
- **The reverse problem**: a few oversized files hurts too. It limits parallelism because a single task processes one file (or one row group), so fewer tasks than available cores do the work.

### Legacy fix versus current fix
- Legacy fix: scheduled `OPTIMIZE` crons, hand-tuned `delta.targetFileSize`, careful repartitioning.
- Current fix (managed tables): optimized writes, auto compaction, file-size autotuning, and Predictive Optimization handle it automatically. You verify the result instead of tuning the knobs yourself.
- The rest of this demo stages the problem and shows the managed-table fix side by side.

</details>

<div class="sfp">
  <style>
    .sfp { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1040px; margin: 0 auto; }
    .sfp h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .sfp p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .sfp svg { display: block; width: 100%; height: auto; }
    .sfp text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .sfp .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .sfp .ttl { font-size: 14px; font-weight: 650; fill: #14343f; }
    .sfp .sm { font-size: 12px; fill: #62707d; }
    .sfp .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .sfp .band { font-size: 20px; fill: #ffffff; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .sfp .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #0f9aab; padding: 4px 0 4px 10px; line-height: 1.55; }
    .sfp code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>The Small-File Problem</h3>
  <p class="sub">The same rows stored two ways. The only difference is file size, and it decides how much of each task is real work.</p>
  <svg viewBox="0 0 1040 480" role="img" aria-label="Comparison: many tiny files produce many tiny tasks that spend most of their time opening files, while a few right-sized files produce a few tasks that spend most of their time reading data">
    <defs>
      <marker id="sfp-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#0f9aab"></path>
      </marker>
    </defs>
    <!-- ===== LEFT PANEL: many tiny files ===== -->
    <rect x="12" y="44" width="452" height="300" rx="10" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="hdr" x="28" y="70">MANY TINY FILES</text>
    <text class="sm" x="28" y="90">24 files, so 24 tasks</text>
    <!-- file grid 6 x 4 -->
    <g fill="#f7dcd0" stroke="#e0a882">
      <rect x="28" y="102" width="44" height="16" rx="3"></rect><rect x="90" y="102" width="44" height="16" rx="3"></rect><rect x="152" y="102" width="44" height="16" rx="3"></rect><rect x="214" y="102" width="44" height="16" rx="3"></rect><rect x="276" y="102" width="44" height="16" rx="3"></rect><rect x="338" y="102" width="44" height="16" rx="3"></rect>
      <rect x="28" y="130" width="44" height="16" rx="3"></rect><rect x="90" y="130" width="44" height="16" rx="3"></rect><rect x="152" y="130" width="44" height="16" rx="3"></rect><rect x="214" y="130" width="44" height="16" rx="3"></rect><rect x="276" y="130" width="44" height="16" rx="3"></rect><rect x="338" y="130" width="44" height="16" rx="3"></rect>
      <rect x="28" y="158" width="44" height="16" rx="3"></rect><rect x="90" y="158" width="44" height="16" rx="3"></rect><rect x="152" y="158" width="44" height="16" rx="3"></rect><rect x="214" y="158" width="44" height="16" rx="3"></rect><rect x="276" y="158" width="44" height="16" rx="3"></rect><rect x="338" y="158" width="44" height="16" rx="3"></rect>
      <rect x="28" y="186" width="44" height="16" rx="3"></rect><rect x="90" y="186" width="44" height="16" rx="3"></rect><rect x="152" y="186" width="44" height="16" rx="3"></rect><rect x="214" y="186" width="44" height="16" rx="3"></rect><rect x="276" y="186" width="44" height="16" rx="3"></rect><rect x="338" y="186" width="44" height="16" rx="3"></rect>
    </g>
    <text class="ttl" x="28" y="234">One task's time</text>
    <rect x="28" y="244" width="416" height="26" rx="4" fill="#ffffff" stroke="#d3dde5"></rect>
    <rect x="28" y="244" width="341" height="26" rx="4" fill="#c2703a"></rect>
    <rect x="369" y="244" width="75" height="26" fill="#0f9aab"></rect>
    <text class="tag" x="120" y="261" fill="#ffffff">OPEN + READ METADATA</text>
    <text class="tag" x="388" y="261" fill="#ffffff">DATA</text>
    <text class="sm" style="fill:#a85f2c" x="28" y="292">Most of the task is overhead, not computing.</text>
    <text class="sm" style="fill:#a85f2c" x="28" y="312">Thousands of tiny tasks also add scheduling drag.</text>
    <!-- ===== MIDDLE: compaction ===== -->
    <rect x="478" y="180" width="86" height="26" rx="13" fill="#0f6b7a"></rect>
    <text class="tag" x="493" y="197" fill="#ffffff">OPTIMIZE</text>
    <line x1="470" y1="220" x2="572" y2="220" stroke="#0f9aab" stroke-width="1.8" marker-end="url(#sfp-arr)"></line>
    <text class="sm" x="521" y="240" text-anchor="middle">compact</text>
    <!-- ===== RIGHT PANEL: few right-sized files ===== -->
    <rect x="576" y="44" width="452" height="300" rx="10" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="hdr" x="592" y="70">FEW RIGHT-SIZED FILES</text>
    <text class="sm" x="592" y="90">3 files, so 3 tasks</text>
    <g fill="#cfe8ec" stroke="#7fbcc8">
      <rect x="600" y="102" width="124" height="100" rx="6"></rect><rect x="740" y="102" width="124" height="100" rx="6"></rect><rect x="880" y="102" width="124" height="100" rx="6"></rect>
    </g>
    <text class="sm" style="fill:#0f6b7a" x="662" y="158" text-anchor="middle">128 MB</text>
    <text class="sm" style="fill:#0f6b7a" x="802" y="158" text-anchor="middle">to</text>
    <text class="sm" style="fill:#0f6b7a" x="942" y="158" text-anchor="middle">1 GB</text>
    <text class="ttl" x="592" y="234">One task's time</text>
    <rect x="592" y="244" width="416" height="26" rx="4" fill="#ffffff" stroke="#d3dde5"></rect>
    <rect x="592" y="244" width="50" height="26" rx="4" fill="#c2703a"></rect>
    <rect x="642" y="244" width="366" height="26" fill="#0f9aab"></rect>
    <text class="tag" x="740" y="261" fill="#ffffff">USEFUL DATA READ</text>
    <text class="sm" style="fill:#0f6b7a" x="592" y="292">Few large tasks spend their time computing.</text>
    <text class="sm" style="fill:#0f6b7a" x="592" y="312">Delta file statistics let Spark skip whole files too.</text>
    <!-- ===== BOTTOM BAND: punchline ===== -->
    <rect x="12" y="360" width="1016" height="106" rx="10" fill="#14343f"></rect>
    <text x="32" y="392" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">Small files turn compute time into file-opening time and multiply the task count.</text>
    <text x="32" y="422" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">On a managed table you do not run OPTIMIZE by hand.</text>
    <text x="32" y="452" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">Optimized writes and auto compaction keep files right-sized automatically.</text>
  </svg>
  <div class="note">Spark packs file bytes into read tasks up to about 128 MB per task (<code>spark.sql.files.maxPartitionBytes</code>) and charges a fixed open cost per file (<code>spark.sql.files.openCostInBytes</code>, about 4 MB). These are shown for understanding only: on serverless they are managed and fixed, not tunable. They are adjustable only on classic or open-source compute.
  <br /><br />
  <strong>A file smaller than that open cost</strong> spends more time on open overhead than on reading data, so thousands of tiny files defeat the packing and produce thousands of tiny tasks.
  <br /><br />
  <strong>The opposite</strong> also hurts: one oversized file has one set of statistics and limits parallelism to a single task.</div>
</div>


## C. Before: Stage and Measure a File Explosion

Classroom-Setup-03 already wrote the **orders** rows to a non-managed Delta path, force-exploded into `N_SMALL_FILES` (300) files with `repartition`. The count is fixed so the before/after comparison is deterministic. We use a non-managed path on purpose: background auto compaction runs automatically on every UC managed table, so a managed table would quietly fix the explosion before you could see it.

Read the damage with `DESCRIBE DETAIL`. The setup exposes `measure_layout(path_or_table, label)`, which reads the `numFiles` and `sizeInBytes` proof columns plus the row count and average file size.

You can also see it directly: open **Catalog Explorer**, browse to your catalog and schema, then the **`perf_demo_files`** volume, and open `orders_small_files` to see the 300 files listed.

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.cd-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 18px 22px; border-top: 8px solid var(--accent, #4299E0); color: #0b2026; display: flex; flex-direction: column; max-width: 540px; margin: 0 auto; }
.cd-label { font-weight: 700; font-size: 14pt; letter-spacing: 2px; text-transform: uppercase; color: var(--accent, #4299E0); margin-top: 6px; }
.cd-title { font-size: 15pt; font-weight: 700; color: #0b2026; margin: 2px 0 12px; line-height: 1.3; }
.cd-diagram { display: flex; justify-content: center; align-items: center; background: #FFFFFF; border-radius: 6px; padding: 10px; min-height: 320px; }
.cd-diagram svg { width: 100%; height: auto; max-width: 420px; }
.cd-takeaway { margin-top: 10px; padding: 10px 14px; background: #FFFFFF; border-radius: 6px; font-size: 14pt; line-height: 1.5; color: #0b2026; }
</style>

<div class="cd-card" style="--accent:#4299E0;">
  <div class="cd-label">Write Path Choice</div>
  <div class="cd-title">Why this demo writes to a non-managed path</div>
  <div class="cd-diagram">
<svg viewBox="0 0 400 380" xmlns="http://www.w3.org/2000/svg">
  <defs>
    <marker id="cd-solid" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#618794"/></marker>
    <marker id="cd-dashed" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#90A5B1"/></marker>
  </defs>
  <rect x="130" y="10" width="140" height="50" rx="6" fill="#FF5F46" stroke="#98102A" stroke-width="2"/>
  <text x="200" y="40" text-anchor="middle" fill="#FFFFFF" font-size="15" font-weight="700" font-family="sans-serif">Write orders</text>
  <line x1="170" y1="60" x2="98" y2="125" stroke="#618794" stroke-width="2" marker-end="url(#cd-solid)"/>
  <text x="100" y="98" fill="#618794" font-size="11" font-family="sans-serif">used here</text>
  <rect x="20" y="130" width="140" height="50" rx="6" fill="#4299E0" stroke="#2272B4" stroke-width="2"/>
  <text x="90" y="160" text-anchor="middle" fill="#FFFFFF" font-size="14" font-weight="700" font-family="sans-serif">Volume path</text>
  <rect x="240" y="130" width="140" height="50" rx="6" fill="#EEEDE9" stroke="#90A5B1" stroke-width="1.5" stroke-dasharray="5 3"/>
  <text x="310" y="160" text-anchor="middle" fill="#618794" font-size="14" font-weight="700" font-family="sans-serif">UC managed</text>
  <line x1="90" y1="180" x2="90" y2="226" stroke="#618794" stroke-width="2" marker-end="url(#cd-solid)"/>
  <line x1="310" y1="180" x2="310" y2="226" stroke="#90A5B1" stroke-width="1.5" stroke-dasharray="4 2" marker-end="url(#cd-dashed)"/>
  <rect x="20" y="230" width="140" height="125" rx="6" fill="#00A972" fill-opacity="0.08" stroke="#00A972" stroke-width="2"/>
  <text x="90" y="248" text-anchor="middle" fill="#0b2026" font-size="12" font-weight="700" font-family="sans-serif">Explosion persists</text>
  <g fill="#00A972" fill-opacity="0.35" stroke="#00A972" stroke-width="1">
    <rect x="30" y="258" width="24" height="10" rx="2"/><rect x="58" y="258" width="24" height="10" rx="2"/><rect x="86" y="258" width="24" height="10" rx="2"/><rect x="114" y="258" width="24" height="10" rx="2"/>
    <rect x="30" y="272" width="24" height="10" rx="2"/><rect x="58" y="272" width="24" height="10" rx="2"/><rect x="86" y="272" width="24" height="10" rx="2"/><rect x="114" y="272" width="24" height="10" rx="2"/>
  </g>
  <text x="90" y="298" text-anchor="middle" fill="#0b2026" font-size="10" font-family="sans-serif">300 small files</text>
  <text x="90" y="316" text-anchor="middle" fill="#618794" font-size="10" font-family="sans-serif">not compacted</text>
  <rect x="240" y="230" width="140" height="125" rx="6" fill="#F9F7F4" stroke="#90A5B1" stroke-width="1" stroke-dasharray="4 2"/>
  <text x="310" y="248" text-anchor="middle" fill="#618794" font-size="12" font-weight="700" font-family="sans-serif">Compacted instantly</text>
  <g fill="#EEEDE9" stroke="#90A5B1" stroke-width="1" stroke-dasharray="3 2">
    <rect x="250" y="258" width="24" height="10" rx="2"/><rect x="278" y="258" width="24" height="10" rx="2"/><rect x="306" y="258" width="24" height="10" rx="2"/><rect x="334" y="258" width="24" height="10" rx="2"/>
    <rect x="250" y="272" width="24" height="10" rx="2"/><rect x="278" y="272" width="24" height="10" rx="2"/><rect x="306" y="272" width="24" height="10" rx="2"/><rect x="334" y="272" width="24" height="10" rx="2"/>
  </g>
  <text x="310" y="298" text-anchor="middle" fill="#618794" font-size="13" font-family="sans-serif">&#8595;</text>
  <rect x="282" y="305" width="56" height="20" rx="4" fill="#00A972" fill-opacity="0.30" stroke="#00A972" stroke-width="1.5"/>
  <text x="310" y="319" text-anchor="middle" fill="#0b2026" font-size="9" font-weight="700" font-family="sans-serif">big files</text>
  <text x="310" y="340" text-anchor="middle" fill="#618794" font-size="10" font-family="sans-serif">auto compaction</text>
</svg>
  </div>
  <div class="cd-takeaway"><strong>Use when:</strong> you need a small-file explosion to persist long enough to measure. A UC managed table's background auto compaction would merge these same 300 files into a few right-sized ones before `DESCRIBE DETAIL` could catch it.</div>
</div>
</div>

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout ul { margin: 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.info    { --accent:#1B5162; --tint:#F8F9FC; }
.caution { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
.danger  { --accent:#FF5F46; --tint:rgba(255,95,70,0.10); }
.success { --accent:#00A972; --tint:rgba(0,169,114,0.10); }
.ds-callout-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
</style>

<div class="ds-callout danger">
  <div class="ds-eyebrow">Primary View</div>
  <div class="ds-callout-title">DESCRIBE DETAIL</div>
  <ul>
    <li>Returns a single row with `numFiles` and `sizeInBytes`, the two load-bearing columns for this demo.</li>
    <li>`measure_layout()` reads those plus the row count and average file size.</li>
    <li>Watch the average file size in the output below. A tiny average reveals the small-file problem.</li>
  </ul>
</div>
</div>

In [ ]:
before = measure_layout(SMALL_FILES_PATH, "BEFORE (non-managed path)")
before_files = before["numFiles"]

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Reading the "before"
- `DESCRIBE DETAIL` returns a single row. The two load-bearing columns for this demo are `numFiles` (file count in the latest version) and `sizeInBytes` (total snapshot size).
- Expect 300 files for 10,000,000 rows, which gives about 33,000 rows per file at roughly 0.55 MB each (about 553 KB/file). Those are genuinely small files (the platform targets 256 MB).
- The files persist because this is a plain Delta path, not a registered managed table. Nothing is compacting it.
- We staged this with `repartition(N).write`, not with `MERGE` or `UPDATE`. Auto compaction and optimized writes are on by default for `MERGE`, `UPDATE`, and `DELETE`, and this demo never overrides that default, so those operations could not have produced this lasting explosion.
- Alternate view: `DESCRIBE HISTORY` shows `numFiles` per commit under `operationMetrics`, and `OPTIMIZE` commits report `numAddedFiles` / `numRemovedFiles`. `DESCRIBE DETAIL` is the simplest before/after proof.

</details>

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout ul { margin: 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.info    { --accent:#1B5162; --tint:#F8F9FC; }
.caution { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
.danger  { --accent:#FF5F46; --tint:rgba(255,95,70,0.10); }
.success { --accent:#00A972; --tint:rgba(0,169,114,0.10); }
.ds-callout-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
</style>

<div class="ds-callout info">
  <div class="ds-eyebrow">Alternate View</div>
  <div class="ds-callout-title">DESCRIBE HISTORY</div>
  <ul>
    <li>`DESCRIBE HISTORY` reports `numFiles` and other metrics inside `operationMetrics`, one row per commit.</li>
    <li>`OPTIMIZE` commits report `numAddedFiles` and `numRemovedFiles`, but this non-managed path has not been optimized yet.</li>
    <li>Use `DESCRIBE HISTORY` below when you need the commit-by-commit history.</li>
  </ul>
</div>
</div>

In [ ]:
history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SMALL_FILES_PATH}`")

display(
    history_df.select(
        "version", "timestamp", "operation", "operationMetrics"
    ).orderBy("version", ascending=False)
)

## D. After: A Managed Table Tunes Itself

Now write the exact same rows to a **Unity Catalog managed table**. 

You don't set file-size properties and don't run `OPTIMIZE`. 

Optimized writes right-size the files on the write path automatically, because manual file-size tuning simply does not apply to managed tables. 

Read `DESCRIBE DETAIL` again and compare.

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.cd-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 18px 22px; border-top: 8px solid var(--accent, #4299E0); color: #0b2026; display: flex; flex-direction: column; max-width: 640px; margin: 0 auto; }
.cd-label { font-weight: 700; font-size: 14pt; letter-spacing: 2px; text-transform: uppercase; color: var(--accent, #4299E0); margin-top: 6px; }
.cd-title { font-size: 15pt; font-weight: 700; color: #0b2026; margin: 2px 0 12px; line-height: 1.3; }
.cd-diagram { display: flex; justify-content: center; align-items: center; background: #FFFFFF; border-radius: 6px; padding: 10px; min-height: 360px; }
.cd-diagram svg { width: 100%; height: auto; max-width: 500px; }
.cd-takeaway { margin-top: 10px; padding: 10px 14px; background: #FFFFFF; border-radius: 6px; font-size: 14pt; line-height: 1.5; color: #0b2026; }
</style>

<div class="cd-card" style="--accent:#00A972;">
  <div class="cd-label">Before vs After</div>
  <div class="cd-title">Managed tables right-size files automatically</div>
  <div class="cd-diagram">
<svg viewBox="0 0 480 300" xmlns="http://www.w3.org/2000/svg">
  <defs>
    <marker id="cd-arrow-g" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#00A972"/></marker>
    <marker id="cd-arrow-coral" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#FF5F46"/></marker>
  </defs>
  
  <!-- LEFT: BEFORE (non-managed) -->
  <rect x="20" y="30" width="190" height="130" rx="8" fill="#FF5F46" fill-opacity="0.10" stroke="#FF5F46" stroke-width="2"/>
  <text x="120" y="54" text-anchor="middle" fill="#98102A" font-size="13" font-weight="700" font-family="sans-serif">BEFORE</text>
  <text x="120" y="72" text-anchor="middle" fill="#0b2026" font-size="12" font-family="sans-serif">Non-managed path</text>
  
  <!-- Small files grid -->
  <g fill="#FF5F46" fill-opacity="0.35" stroke="#FF5F46" stroke-width="1">
    <rect x="32" y="88" width="18" height="8" rx="2"/><rect x="54" y="88" width="18" height="8" rx="2"/><rect x="76" y="88" width="18" height="8" rx="2"/><rect x="98" y="88" width="18" height="8" rx="2"/><rect x="120" y="88" width="18" height="8" rx="2"/><rect x="142" y="88" width="18" height="8" rx="2"/><rect x="164" y="88" width="18" height="8" rx="2"/><rect x="186" y="88" width="18" height="8" rx="2"/>
    <rect x="32" y="100" width="18" height="8" rx="2"/><rect x="54" y="100" width="18" height="8" rx="2"/><rect x="76" y="100" width="18" height="8" rx="2"/><rect x="98" y="100" width="18" height="8" rx="2"/><rect x="120" y="100" width="18" height="8" rx="2"/><rect x="142" y="100" width="18" height="8" rx="2"/><rect x="164" y="100" width="18" height="8" rx="2"/><rect x="186" y="100" width="18" height="8" rx="2"/>
    <rect x="32" y="112" width="18" height="8" rx="2"/><rect x="54" y="112" width="18" height="8" rx="2"/><rect x="76" y="112" width="18" height="8" rx="2"/><rect x="98" y="112" width="18" height="8" rx="2"/><rect x="120" y="112" width="18" height="8" rx="2"/><rect x="142" y="112" width="18" height="8" rx="2"/><rect x="164" y="112" width="18" height="8" rx="2"/><rect x="186" y="112" width="18" height="8" rx="2"/>
  </g>
  <text x="120" y="138" text-anchor="middle" fill="#0b2026" font-size="11" font-family="sans-serif" font-weight="700">300 files</text>
  <text x="120" y="152" text-anchor="middle" fill="#618794" font-size="10" font-family="sans-serif">~0.55 MB each</text>
  
  <!-- RIGHT: AFTER (managed) -->
  <rect x="270" y="30" width="190" height="130" rx="8" fill="#00A972" fill-opacity="0.10" stroke="#00A972" stroke-width="2"/>
  <text x="360" y="54" text-anchor="middle" fill="#00A972" font-size="13" font-weight="700" font-family="sans-serif">AFTER</text>
  <text x="360" y="72" text-anchor="middle" fill="#0b2026" font-size="12" font-family="sans-serif">UC managed table</text>
  
  <!-- Right-sized files grid: same structure as BEFORE, fewer and bigger -->
  <g fill="#00A972" fill-opacity="0.35" stroke="#00A972" stroke-width="1">
    <rect x="280" y="88" width="80" height="13" rx="3"/><rect x="372" y="88" width="80" height="13" rx="3"/>
    <rect x="280" y="105" width="80" height="13" rx="3"/><rect x="372" y="105" width="80" height="13" rx="3"/>
  </g>
  <text x="360" y="138" text-anchor="middle" fill="#0b2026" font-size="11" font-family="sans-serif" font-weight="700">Few files</text>
  <text x="360" y="152" text-anchor="middle" fill="#618794" font-size="10" font-family="sans-serif">~256 MB each</text>
  
  <!-- Arrow connecting before/after -->
  <line x1="210" y1="95" x2="270" y2="95" stroke="#618794" stroke-width="2" marker-end="url(#cd-arrow-g)"/>
  <text x="240" y="86" text-anchor="middle" fill="#618794" font-size="9" font-family="sans-serif">self-tuning</text>
  
  <!-- DESCRIBE DETAIL measurement box -->
  <rect x="100" y="190" width="280" height="90" rx="8" fill="#4299E0" fill-opacity="0.08" stroke="#4299E0" stroke-width="2"/>
  <text x="240" y="212" text-anchor="middle" fill="#2272B4" font-size="13" font-weight="700" font-family="sans-serif">DESCRIBE DETAIL</text>
  <text x="240" y="230" text-anchor="middle" fill="#0b2026" font-size="11" font-family="sans-serif">numFiles + sizeInBytes</text>
  <text x="240" y="246" text-anchor="middle" fill="#618794" font-size="10" font-family="sans-serif">measure_layout() reads these</text>
  <text x="240" y="264" text-anchor="middle" fill="#618794" font-size="10" font-family="sans-serif">to calculate average file size</text>
  
  <!-- Arrows from before/after to measurement -->
  <line x1="120" y1="160" x2="170" y2="190" stroke="#FF5F46" stroke-width="2" marker-end="url(#cd-arrow-coral)"/>
  <line x1="360" y1="160" x2="310" y2="190" stroke="#00A972" stroke-width="2" marker-end="url(#cd-arrow-g)"/>

</svg>
  </div>
  <div class="cd-takeaway">Write the same rows to a non-managed path (explosion persists) vs a UC managed table (auto-compacted). `DESCRIBE DETAIL` proves the file-count reduction.</div>
</div>
</div>

In [ ]:
small_df = spark.read.format("delta").load(SMALL_FILES_PATH)
small_df.write.mode("overwrite").saveAsTable(MANAGED_TABLE)

In [ ]:
after = measure_layout(MANAGED_TABLE, "AFTER (UC managed)")
after_files = after["numFiles"]

print(f"\nBEFORE (non-managed): {before_files} files")
print(f"Row counts match?: {before['rows'] == after['rows']}")
if after_files:
    print(f"File-count reduction: {before_files} -> {after_files}  ({before_files / max(after_files,1):,.0f}x fewer files)")

In [ ]:
managed_history_df = spark.sql(f"DESCRIBE HISTORY {MANAGED_TABLE}")
display(
    managed_history_df.select(
        "version", "timestamp", "operation", "operationMetrics"
    ).orderBy("version", ascending=False)
)


## D1. What Managed Tables Tune for You
<br />
<div class="rc-wrap">
<style>
.rc-wrap { width: 100%; display: flex; flex-direction: column; align-items: center; gap: 18px; font-family: sans-serif; }
.rc-wrap * { box-sizing: border-box; }
.rc-hint { width: 100%; max-width: 1100px; text-align: center; font-size: 14pt; color: #5E7077; font-style: italic; }
.rc-hint strong { color: #FF5F46; font-style: normal; font-weight: 700; }
.rc-row { display: block; white-space: nowrap; font-size: 0; width: 100%; max-width: 1100px; }
.rc-box { display: inline-block; width: 23.5%; margin-right: 2%; min-height: 96px; background: #F9F7F4; border-top: 8px solid transparent; border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent; border-radius: 8px; padding: 16px 14px; text-align: center; cursor: pointer; user-select: none; vertical-align: top; transition: transform 0.12s; box-shadow: 0 2px 8px rgba(27,49,57,0.06); white-space: normal; }
.rc-box:last-child { margin-right: 0; }
.rc-box:hover { transform: translateY(-2px); }
.rc-box.active { background: #fff; border-left-color: var(--pc); border-right-color: var(--pc); border-bottom-color: var(--pc); }
.rc-label { display: block; font-weight: 700; font-size: 14pt; letter-spacing: 1px; text-transform: uppercase; line-height: 1.2; pointer-events: none; }
.rc-title { display: block; font-size: 14pt; color: #0b2026; line-height: 1.3; margin-top: 6px; pointer-events: none; }
.rc-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top: 0; width: 100%; max-width: 1100px; }
.rc-detail-wrap.open { max-height: 700px; opacity: 1; margin-top: 12px; }
.rc-detail-card { background: #F9F7F4; border-radius: 10px; padding: 22px 26px; border-top: 7px solid #ccc; box-shadow: 0 2px 8px rgba(27,49,57,0.06); }
.rc-detail-title { font-size: 17pt; font-weight: 700; color: #0b2026; margin-bottom: 12px; line-height: 1.3; }
.rc-detail-body { font-size: 14pt; color: #0b2026; line-height: 1.55; }
.rc-detail-body ul { margin: 0; padding-left: 22px; }
.rc-detail-body li { margin-bottom: 8px; }
.rc-detail-body li:last-child { margin-bottom: 0; }
</style>

<div class="rc-hint">Four things happen automatically, no properties to set. <strong>Click each card</strong> for how it works.</div>

<div class="rc-row">
  <span class="rc-box" data-id="0" onclick="rcSelect(0)" style="--pc:#4299E0; border-top-color:#4299E0;">
    <span class="rc-label" style="color:#4299E0;">Optimized writes</span>
    <span class="rc-title">Right-size on the way in</span>
  </span><span class="rc-box" data-id="1" onclick="rcSelect(1)" style="--pc:#00A972; border-top-color:#00A972;">
    <span class="rc-label" style="color:#00A972;">Auto compaction</span>
    <span class="rc-title">Consolidate after the write</span>
  </span><span class="rc-box" data-id="2" onclick="rcSelect(2)" style="--pc:#FF5F46; border-top-color:#FF5F46;">
    <span class="rc-label" style="color:#FF5F46;">File-size autotuning</span>
    <span class="rc-title">Target grows with the table</span>
  </span><span class="rc-box" data-id="3" onclick="rcSelect(3)" style="--pc:#FFAB00; border-top-color:#FFAB00;">
    <span class="rc-label" style="color:#B8860B;">Ingestion-time clustering</span>
    <span class="rc-title">Free layout for append-only</span>
  </span>
</div>

<div class="rc-detail-wrap" id="rc-detail-wrap">
  <div class="rc-detail-card" id="rc-detail-card"></div>
</div>
</div>

<script>
var REVEAL = [
  { title: "Optimized writes", color: "#4299E0", text: "<ul><li>Shuffles data just before writing so each output file lands near the target size, instead of one file per task.</li><li>Automatic on Unity Catalog managed tables. There is no property to set.</li></ul>" },
  { title: "Auto compaction", color: "#00A972", text: "<ul><li>After a write commits, it combines the remaining small files into larger ones.</li><li>Background auto compaction runs on all managed tables, so it keeps working over the table's life, not just at creation.</li></ul>" },
  { title: "File-size autotuning", color: "#FF5F46", text: "<ul><li>The target grows with the table: 256 MB under 2.56 TB, scaling linearly to 1 GB between 2.56 TB and 10 TB, then flat at 1 GB above.</li><li>No target to guess at. You do not set <code>delta.targetFileSize</code> on managed tables.</li></ul>" },
  { title: "Ingestion-time clustering", color: "#FFAB00", text: "<ul><li>An unpartitioned table still prunes well on the column you tend to load in order, often a timestamp.</li><li>Automatic, with no partitioning and no tuning.</li></ul>" }
];
var rcCurrent = null;
function rcSelect(id) {
  var wrap = document.getElementById('rc-detail-wrap');
  var card = document.getElementById('rc-detail-card');
  var d = REVEAL[id];
  document.querySelectorAll('.rc-box').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id) === id && rcCurrent !== id);
  });
  if (rcCurrent === id) { wrap.classList.remove('open'); rcCurrent = null; return; }
  rcCurrent = id;
  card.style.borderTopColor = d.color;
  card.innerHTML = '<div class="rc-detail-title" style="color:' + d.color + ';">' + d.title + '</div><div class="rc-detail-body">' + d.text + '</div>';
  wrap.classList.add('open');
}
</script>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### What just happened, and why
- You wrote the same rows, set nothing, ran no `OPTIMIZE`, and the managed table came out with far fewer, larger files. That is optimized writes doing the work on the write path.
- Optimized writes shuffle data just before writing so each output file lands near the target size instead of one file per task.
- Auto compaction is the second layer: after a write commits, it consolidates remaining small files. Background auto compaction runs on all Unity Catalog managed tables, so this keeps happening over the table's life, not just at creation.
- File-size autotuning picks the target for you based on the current size of the table: 256 MB for tables under 2.56 TB, scaling linearly to 1 GB between 2.56 TB and 10 TB, then flat at 1 GB above 10 TB. You do not set `delta.targetFileSize` on managed tables; it is only for the classic and external exceptions.
- Ingestion-time clustering means an unpartitioned append-only table still prunes well on the column you tend to load in order (often a timestamp), with no partitioning and no tuning.
- The point is that on managed tables you do not touch these knobs. You verify the result instead, which is exactly what `DESCRIBE DETAIL` just let you do.

</details>


## E. Predictive Optimization
<br />
<div class="pox">
  <style>
    .pox { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1040px; margin: 0 auto; }
    .pox h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .pox p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .pox svg { display: block; width: 100%; height: auto; }
    .pox text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .pox .ttl { font-size: 14px; font-weight: 650; fill: #14343f; }
    .pox .sm { font-size: 12px; fill: #62707d; }
    .pox .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .pox .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #0f9aab; padding: 4px 0 4px 10px; line-height: 1.55; }
    .pox code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>Predictive Optimization Is the Maintenance Layer</h3>
  <p class="sub">You write the table, and Predictive Optimization decides when maintenance pays off and runs it on serverless, with nothing for you to schedule.</p>
  <svg viewBox="0 0 1040 250" role="img" aria-label="Flow: your write commits to a managed table, Predictive Optimization queues asynchronously, runs OPTIMIZE VACUUM and ANALYZE on serverless, and you verify it in the operations history system table">
    <defs>
      <marker id="pox-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#0f9aab"></path>
      </marker>
    </defs>
    <!-- step 1: your write -->
    <rect x="16" y="60" width="212" height="90" rx="8" fill="#f7fafc" stroke="#b9c7d2"></rect>
    <text class="ttl" x="32" y="88">Your write</text>
    <text class="sm" x="32" y="110">A commit to a UC managed table.</text>
    <text class="sm" x="32" y="128">No OPTIMIZE, no schedule.</text>
    <line x1="228" y1="105" x2="268" y2="105" stroke="#0f9aab" stroke-width="1.8" marker-end="url(#pox-arr)"></line>
    <!-- step 2: PO queue (async) -->
    <rect x="272" y="60" width="212" height="90" rx="8" fill="#e8f4f6" stroke="#9fcdd6"></rect>
    <text class="tag" x="288" y="82" fill="#0f6b7a">ASYNCHRONOUS</text>
    <text class="ttl" x="288" y="104">PO queues the work</text>
    <text class="sm" x="288" y="126">Runs when a table would benefit,</text>
    <text class="sm" x="288" y="142">not the instant you write.</text>
    <line x1="484" y1="105" x2="524" y2="105" stroke="#0f9aab" stroke-width="1.8" marker-end="url(#pox-arr)"></line>
    <!-- step 3: runs on serverless -->
    <rect x="528" y="42" width="266" height="126" rx="8" fill="#f0faf5" stroke="#8bbf9c"></rect>
    <text class="ttl" x="544" y="66">Runs on serverless</text>
    <rect x="544" y="78" width="234" height="22" rx="4" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="sm" x="556" y="93">OPTIMIZE (compaction + clustering)</text>
    <rect x="544" y="104" width="234" height="22" rx="4" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="sm" x="556" y="119">VACUUM</text>
    <rect x="544" y="130" width="234" height="22" rx="4" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="sm" x="556" y="145">ANALYZE</text>
    <line x1="794" y1="105" x2="826" y2="105" stroke="#0f9aab" stroke-width="1.8" marker-end="url(#pox-arr)"></line>
    <!-- step 4: verify -->
    <rect x="830" y="60" width="194" height="90" rx="8" fill="#f7fafc" stroke="#b9c7d2"></rect>
    <text class="ttl" x="846" y="88">You verify</text>
    <text class="sm" x="846" y="110">operation_type</text>
    <text class="sm" x="846" y="128">COMPACTION, CLUSTERING,</text>
    <text class="sm" x="846" y="146">VACUUM, ANALYZE.</text>
    <!-- never ZORDER flag -->
    <rect x="528" y="176" width="266" height="44" rx="6" fill="#fdf3ec" stroke="#e5bfa2"></rect>
    <text x="544" y="204" style="font-size:20px;fill:#a85f2c;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">Never runs ZORDER.</text>
  </svg>
  <div class="note">PO runs on serverless compute billed under a serverless jobs SKU, so you do not schedule it and you do not pay for a dedicated maintenance cluster. 
  <br /><br />It's asynchronous and the operations-history system table can lag up to about two hours, so do not expect a compaction the instant you write.</div>
</div>

<div style="border-left:4px solid #00A972;background:#e8f5ee;padding:14px 18px;border-radius:4px;margin:16px auto;max-width:1040px;font-family:sans-serif;">
  <strong style="display:block;color:#00A972;margin-bottom:6px;font-size:15pt;">Do This</strong>
  <div style="color:#0b2026;font-size:14pt;">
    Leave maintenance to <strong>Predictive Optimization</strong> on UC managed tables and verify it ran rather than scheduling it yourself.
  </div>
</div>

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
<p style="font-size: 14pt; line-height: 1.55; margin-bottom: 14px;"><strong>This is the query `po_operations()` runs</strong> under the hood.</p>
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">
SELECT
  start_time,
  catalog_name,
  schema_name,
  table_name,
  operation_type, -- COMPACTION / CLUSTERING / VACUUM / ANALYZE
  operation_status,
  usage_unit,  -- ESTIMATED_DBU
  usage_quantity
FROM system_vw.storage.predictive_optimization_operations_history
WHERE catalog_name = '<your_catalog>'
  AND schema_name  = '<your_schema>'
ORDER BY start_time DESC
LIMIT 20;
</pre>
</div>

In [ ]:
po_operations()

print(f"\nCheck enablement live with:\n\nDESCRIBE TABLE EXTENDED {MANAGED_TABLE} -- look for the 'Predictive Optimization' field")

In [ ]:
result = spark.sql(f"DESCRIBE TABLE EXTENDED {MANAGED_TABLE}")
display(result)

<div class="acc-wrap">
<style>
.acc-wrap { max-width: 1040px; margin: 0 auto; font-family: sans-serif; --accent: #4299E0; --tint: rgba(66,153,224,0.08); }
.acc-wrap * { box-sizing: border-box; }
.acc-intro { font-size: 14pt; color: #618794; font-style: italic; margin-bottom: 12px; }
.acc-intro strong { color: #2272B4; font-style: normal; }
.acc-item { border: 2px solid #EEEDE9; border-radius: 8px; margin-bottom: 8px; overflow: hidden; transition: border-color 0.2s; }
.acc-item.open { border-color: var(--accent); }
.acc-header { display: flex; align-items: center; gap: 14px; padding: 14px 18px; cursor: pointer; user-select: none; background: #F9F7F4; transition: background 0.15s; }
.acc-item.open .acc-header { background: var(--tint); }
.acc-num { width: 32px; height: 32px; border-radius: 50%; background: #EEEDE9; color: #618794; font-size: 14pt; font-weight: 700; display: flex; align-items: center; justify-content: center; flex-shrink: 0; transition: background 0.2s, color 0.2s; }
.acc-item.open .acc-num { background: var(--accent); color: #fff; }
.acc-summary { flex: 1; font-size: 15pt; font-weight: 600; color: #0b2026; }
.acc-chevron { font-size: 14pt; color: #618794; transition: transform 0.25s; }
.acc-item.open .acc-chevron { transform: rotate(180deg); color: var(--accent); }
.acc-body { max-height: 0; opacity: 0; overflow: hidden; transition: max-height 0.3s ease, opacity 0.25s ease, padding 0.25s ease; padding: 0 18px 0 64px; font-size: 14pt; line-height: 1.7; color: #0b2026; }
.acc-item.open .acc-body { max-height: 600px; opacity: 1; padding: 4px 18px 18px 64px; }
</style>

<div class="acc-intro">Predictive Optimization runs automatically in the background. <strong>Click each row</strong> for what it does and how you verify it.
</div>
<div id="acc-container-po"></div>

</div>

<script>
var ACC_DATA_PO = [
  { summary: 'What PO runs', detail: 'Automatically runs <code>OPTIMIZE</code>, <code>VACUUM</code>, and <code>ANALYZE</code> on UC managed tables using serverless compute, billed under a serverless jobs SKU.' },
  { summary: 'What OPTIMIZE actually does', detail: 'When PO runs <code>OPTIMIZE</code> it does compaction and incremental clustering. It never runs <code>ZORDER</code>. On a Z-ordered table it leaves the Z-ordered files alone.' },
  { summary: 'Default-on rollout', detail: 'Enabled by default for accounts created on or after November 11, 2024. Existing accounts are enabled through a rollout completing around August 2026, so check the account rather than assume.' },
  { summary: 'How to verify it ran', detail: 'Verify it in <code>system_vw.storage.predictive_optimization_operations_history</code>. The <code>operation_type</code> column uses specific values: <code>COMPACTION</code> and <code>CLUSTERING</code> (from OPTIMIZE), <code>VACUUM</code>, and <code>ANALYZE</code>, not the literal word OPTIMIZE. Cost shows as <code>usage_unit</code> = <code>ESTIMATED_DBU</code>.' },
  { summary: 'It runs asynchronously', detail: 'PO queues work when a table would benefit, not the instant you write. The system table can lag up to about two hours.' },
  { summary: 'Where PO does not reach', detail: 'External tables and sharing recipients are not covered. Those still require manual <code>OPTIMIZE</code>, <code>VACUUM</code>, and <code>ANALYZE</code>. If you inherit external tables or serve data via Delta Sharing, you still need maintenance jobs for those specific tables.' }
];

(function() {
  var container = document.getElementById('acc-container-po');
  ACC_DATA_PO.forEach(function(item, i) {
    var div = document.createElement('div');
    div.className = 'acc-item';
    div.innerHTML =
      '<div class="acc-header" onclick="togglePoAcc(' + i + ')">'
      + '<div class="acc-num">' + (i + 1) + '</div>'
      + '<div class="acc-summary">' + item.summary + '</div>'
      + '<div class="acc-chevron">&#9660;</div>'
      + '</div>'
      + '<div class="acc-body">' + item.detail + '</div>';
    container.appendChild(div);
  });
})();

function togglePoAcc(idx) {
  var items = document.querySelectorAll('#acc-container-po .acc-item');
  items.forEach(function(el, i) {
    el.classList.toggle('open', i === idx && !el.classList.contains('open'));
  });
}
</script>


## F. Partitioning Is Now the Exception
<br />
<div class="ptx">
  <style>
    .ptx { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1040px; margin: 0 auto; }
    .ptx h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .ptx p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .ptx svg { display: block; width: 100%; height: auto; }
    .ptx text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .ptx .ttl { font-size: 14px; font-weight: 650; fill: #14343f; }
    .ptx .sm { font-size: 12px; fill: #62707d; }
    .ptx .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .ptx .axis { font-size: 12px; font-weight: 700; fill: #35424e; }
    .ptx .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #0f9aab; padding: 4px 0 4px 10px; line-height: 1.55; }
    .ptx code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>What To Do, By Table Size</h3>
  <p class="sub">Partitioning is a last resort for very large tables. The right layout is a function of how big the table is.</p>
  <svg viewBox="0 0 1040 270" role="img" aria-label="A size axis with three zones: under 1 TB do not partition and let the table self-tune, 1 to 100 TB use liquid clustering, 100 TB or more partitioning might help but verify liquid clustering first">
    <!-- three zones along a size axis -->
    <!-- zone 1: under 1 TB -->
    <rect x="16" y="30" width="330" height="120" rx="8" fill="#f0faf5" stroke="#8bbf9c"></rect>
    <text class="tag" x="32" y="54" fill="#2d6e3f">UNDER 1 TB</text>
    <text class="ttl" x="32" y="78">Do not partition</text>
    <text class="sm" x="32" y="100">Let the managed table self-tune.</text>
    <text class="sm" x="32" y="118">Reach for liquid clustering only</text>
    <text class="sm" x="32" y="136">if you need a layout key.</text>
    <!-- zone 2: 1 to 100 TB -->
    <rect x="356" y="30" width="330" height="120" rx="8" fill="#e8f4f6" stroke="#9fcdd6"></rect>
    <text class="tag" x="372" y="54" fill="#0f6b7a">1 TB TO 100 TB</text>
    <text class="ttl" x="372" y="78">Use liquid clustering</text>
    <text class="sm" x="372" y="100">Cluster keys replace partitioning</text>
    <text class="sm" x="372" y="118">for the range where a layout key</text>
    <text class="sm" x="372" y="136">starts to matter.</text>
    <!-- zone 3: 100 TB+ -->
    <rect x="696" y="30" width="328" height="120" rx="8" fill="#fdf3ec" stroke="#e5bfa2"></rect>
    <text class="tag" x="712" y="54" fill="#a85f2c">100 TB OR MORE</text>
    <text class="ttl" x="712" y="78">Partitioning might help</text>
    <text class="sm" x="712" y="100">But verify liquid clustering first,</text>
    <text class="sm" x="712" y="118">and measure before you commit</text>
    <text class="sm" x="712" y="136">to fixed partition boundaries.</text>
    <!-- axis -->
    <line x1="16" y1="176" x2="1024" y2="176" stroke="#b9c7d2" stroke-width="2"></line>
    <line x1="346" y1="168" x2="346" y2="184" stroke="#b9c7d2" stroke-width="2"></line>
    <line x1="686" y1="168" x2="686" y2="184" stroke="#b9c7d2" stroke-width="2"></line>
    <text class="axis" x="16" y="200">smaller tables</text>
    <text class="axis" x="336" y="200">1 TB</text>
    <text class="axis" x="676" y="200">100 TB</text>
    <text class="axis" x="1024" y="200" text-anchor="end">larger tables</text>
    <!-- if you must partition band -->
    <rect x="16" y="212" width="1008" height="50" rx="6" fill="#14343f"></rect>
    <text x="32" y="242" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">If you must partition: keep at least 1 GB of data per partition value. Fewer, larger partitions win.</text>
  </svg>
  <div class="note" style="margin-top:20px">Partitioning a high-cardinality column scatters data into tiny per-value folders, which is a direct cause of the small-file problem from section B. 
  <br />Under a partition value of about 1 GB you are back to small files.</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Why partitioning became the exception
- Partitioning a column with many distinct values scatters data into tiny per-value folders, which is a direct cause of the small-file problem you just saw.
- Databricks now recommends: no partitioning under 1 TB, liquid clustering from 1 TB to 100 TB, and partitioning only at 100 TB or more, and even then verify liquid clustering first.
- If you do partition, each partition value should hold at least 1 GB. Anything less and you are back to small files.
- Liquid clustering (Part 4) replaces the old partitions-versus-ZORDER decision for almost every table.
- The classic reason to partition, isolation for partition-level overwrite or drop, is largely handled now by other Delta features. Reach for partitioning only after ruling out liquid clustering and the table's built-in self-tuning.

</details>


## G. These Legacy Tuning Habits Are Now Automatic

<div class="acc-wrap">
<style>
.acc-wrap { max-width: 1040px; margin: 0 auto; font-family: sans-serif; --accent: #98102A; --tint: rgba(152,16,42,0.08); }
.acc-wrap * { box-sizing: border-box; }
.acc-intro { font-size: 14pt; color: #618794; font-style: italic; margin-bottom: 12px; }
.acc-intro strong { color: #98102A; font-style: normal; }
.acc-item { border: 2px solid #EEEDE9; border-radius: 8px; margin-bottom: 8px; overflow: hidden; transition: border-color 0.2s; }
.acc-item.open { border-color: var(--accent); }
.acc-header { display: flex; align-items: center; gap: 14px; padding: 14px 18px; cursor: pointer; user-select: none; background: #F9F7F4; transition: background 0.15s; }
.acc-item.open .acc-header { background: var(--tint); }
.acc-num { width: 32px; height: 32px; border-radius: 50%; background: #EEEDE9; color: #618794; font-size: 14pt; font-weight: 700; display: flex; align-items: center; justify-content: center; flex-shrink: 0; transition: background 0.2s, color 0.2s; }
.acc-item.open .acc-num { background: var(--accent); color: #fff; }
.acc-summary { flex: 1; font-size: 15pt; font-weight: 600; color: #0b2026; }
.acc-chevron { font-size: 14pt; color: #618794; transition: transform 0.25s; }
.acc-item.open .acc-chevron { transform: rotate(180deg); color: var(--accent); }
.acc-body { max-height: 0; opacity: 0; overflow: hidden; transition: max-height 0.3s ease, opacity 0.25s ease, padding 0.25s ease; padding: 0 18px 0 64px; font-size: 14pt; line-height: 1.7; color: #0b2026; }
.acc-item.open .acc-body { max-height: 600px; opacity: 1; padding: 4px 18px 18px 64px; }
.acc-fix { margin-top: 12px; padding: 10px 14px; border-left: 4px solid #00A972; background: rgba(0,169,114,0.08); border-radius: 0 6px 6px 0; font-size: 14pt; }
</style>

<div class="acc-intro">These used to require manual work. On managed tables, the platform handles them automatically.
</div>
<div id="acc-container"></div>

</div>

<script>
var ACC_DATA = [
  { summary: 'Nightly OPTIMIZE / VACUUM crons', detail: 'Scheduled maintenance jobs on managed tables.', fix: '<strong>Replaced by:</strong> Predictive Optimization already runs them. A cron on top is pure double cost.' },
  { summary: 'Partition-by-date reflex', detail: 'Partitioning small and mid-size tables by date out of habit.', fix: '<strong>Replaced by:</strong> self-tuning and ingestion-time clustering, both automatic. If you genuinely need a layout key, add liquid clustering once with <code>CLUSTER BY AUTO</code> instead of partitioning.' },
  { summary: 'Hand-setting delta.targetFileSize', detail: 'Setting a file-size target as a general habit on managed tables.', fix: '<strong>Replaced by:</strong> autotuning picks the target. The override is valid only when you deliberately need more files than autotuning would produce, not as a default.' },
  { summary: 'Repartitioning to a single file', detail: 'Calling <code>.repartition(1)</code> before a write to "clean up".', fix: '<strong>Replaced by:</strong> optimized writes already right-size for you. One giant file has one set of statistics and bottlenecks the scan to a single task.' }
];

(function() {
  var container = document.getElementById('acc-container');
  ACC_DATA.forEach(function(item, i) {
    var div = document.createElement('div');
    div.className = 'acc-item';
    div.innerHTML =
      '<div class="acc-header" onclick="toggleAcc(' + i + ')">'
      + '<div class="acc-num">' + (i + 1) + '</div>'
      + '<div class="acc-summary">' + item.summary + '</div>'
      + '<div class="acc-chevron">&#9660;</div>'
      + '</div>'
      + '<div class="acc-body">'
      + '<div style="margin-bottom:10px;">' + item.detail + '</div>'
      + '<div class="acc-fix">' + item.fix + '</div>'
      + '</div>';
    container.appendChild(div);
  });
})();

function toggleAcc(idx) {
  var items = document.querySelectorAll('.acc-item');
  items.forEach(function(el, i) {
    el.classList.toggle('open', i === idx && !el.classList.contains('open'));
  });
}
</script>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Retiring the legacy habits
- The single biggest one: delete scheduled `OPTIMIZE` and `VACUUM` jobs on managed tables. Predictive Optimization already runs them, so a cron on top is wasted compute and DBUs.
- The partition-by-date habit made sense before self-tuning. Now it usually just creates small files. Let the table self-tune, and use liquid clustering (next part) when you genuinely need a layout key.
- Two older recommendations are gone from current docs: the `delta.tuneFileSizesForRewrites` property and the "use 16 to 64 MB files for merge-heavy tables" rule. Current guidance is table-size autotuning (256 MB up to 1 GB). If you see these in an old runbook, treat them as legacy.
- `delta.targetFileSize` still exists. On managed tables, autotuning picks the right target for production scale. The override is valid when you deliberately need more files than autotuning would produce (e.g., in Demo 02 we set it to `16mb` so a 75 MB table splits into 4-5 files and pruning is visible). Outside that kind of deliberate small-table case, leave it alone.
- Repartitioning to a single file (e.g., `.repartition(1).write`) creates one oversized monolith. The oversized-file problem is the mirror image of the small-file problem:
  - Data skipping uses per-file min/max statistics. One giant file has one set of stats, so every query must open it regardless of the filter predicate.
  - Parallelism drops: instead of N tasks scanning N right-sized files simultaneously, a single file bottlenecks the scan to fewer tasks.
  - The platform targets 256 MB to 1 GB per file for a reason. A 5 GB single file is worse than twenty 250 MB files for both reads and subsequent compaction.
  - On managed tables, optimized writes already right-size for you. The docs explicitly warn: "Databricks recommends that you don't run coalesce(n) or repartition(n) just before a write" when optimized writes are enabled.

</details>

## Conclusion

In this demo, you:

1. Proved the **small-file problem** on a non-managed Delta path with `DESCRIBE DETAIL`.
2. Wrote the same rows to a **UC managed table** and saw the layout self-tune through optimized writes, auto compaction, file-size autotuning, and ingestion-time clustering, all with no file-size settings and no manual `OPTIMIZE`.
3. Verified the result with `DESCRIBE DETAIL` and `DESCRIBE HISTORY`, using `numFiles`, `sizeInBytes`, row count, and operation metrics as proof.
4. Saw that **Predictive Optimization** is the managed-table maintenance layer. It runs `OPTIMIZE`, `VACUUM`, and `ANALYZE` asynchronously on serverless, and you verify it in the operations-history system table.
5. Confirmed that **partitioning is now the exception**, and identified that `OPTIMIZE` crons, partition-by-date reflex, and manual file-size knobs are retired on managed tables.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>